In [28]:
from deck import full_euchre_deck
from dealer import Dealer
# from numba import njit
from n_branches import array_set_difference
from n_play_round import round1, next_round
from tree_search import player_best_prune, find_best_opener, n_trick_sim, trick_played, find_best_response
import numpy as np


In [46]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands_test = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands_test

array([[[  0, 120],
        [  0, -14],
        [-12,   0],
        [  9,   0],
        [  0,  -9]],

       [[  0, -10],
        [ 10,   0],
        [-11,   0],
        [ 14,   0],
        [  0, 140]],

       [[ 12,   0],
        [  0, 110],
        [-14,   0],
        [ 13,   0],
        [  0, 130]],

       [[  0, -12],
        [  0, 135],
        [ 11,   0],
        [  0, 100],
        [  0,  90]]])

In [47]:
# hands_test = np.array([[[-10,   0],
#         [-11,   0],
#         [-14,   0],
#         [-12,   0],
#         [  0, -10]],

#        [[  0, -12],
#         [  0, -13],
#         [  0, -14],
#         [-13,   0],
#         [  0, 100]],

#        [[ 14,   0],
#         [  0, 135],
#         [ 10,   0],
#         [  0,  90],
#         [  0,  -9]],

#        [[ 12,   0],
#         [  0, 140],
#         [ 11,   0],
#         [  9,   0],
#         [  0, 110]]])

In [48]:
def definitive_winner(dealt_hands, starting_player, verbose):

    # round 1
    best_opener = find_best_opener(
        lead=starting_player,
        hands=dealt_hands,
        tricks=5,
        previous_winners=np.array([]),
        sim_func=n_trick_sim,
        verbose=verbose,
    )

    r2_hand, r2_lead = find_best_response(
        lead=starting_player,
        hands=dealt_hands,
        tricks=5,
        previous_winners=np.array([], dtype=np.int64),
        best_opener=best_opener,
    )

    # round 2

    best_opener_r2 = find_best_opener(
        lead=r2_lead,
        hands=r2_hand,
        tricks=4,
        previous_winners=np.array([r2_lead]),
        sim_func=n_trick_sim,
        verbose=verbose,
    )
    r3_hand, r3_lead = find_best_response(
        lead=r2_lead,
        hands=r2_hand,
        tricks=4,
        previous_winners=np.array([r2_lead], dtype=np.int64),
        best_opener=best_opener_r2,
    )

    # round 3
    best_opener_r3 = find_best_opener(
        lead=r3_lead,
        hands=r3_hand,
        tricks=3,
        previous_winners=np.array([r2_lead, r3_lead]),
        sim_func=n_trick_sim,
        verbose=verbose,
    )
    r4_hand, r4_lead = find_best_response(
        lead=r3_lead,
        hands=r3_hand,
        tricks=3,
        previous_winners=np.array([r2_lead, r3_lead], dtype=np.int64),
        best_opener=best_opener_r3,
    )

    # round 4

    best_opener_r4 = find_best_opener(
        lead=r4_lead,
        hands=r4_hand,
        tricks=2,
        previous_winners=np.array([r2_lead, r3_lead, r4_lead]),
        sim_func=n_trick_sim,
        verbose=verbose,
    )
    r5_hand, r5_lead = find_best_response(
        lead=r4_lead,
        hands=r4_hand,
        tricks=2,
        previous_winners=np.array([r2_lead, r3_lead, r4_lead], dtype=np.int64),
        best_opener=best_opener_r4,
    )


    # round 5

    best_opener_r5 = find_best_opener(
        lead=r5_lead,
        hands=r5_hand,
        tricks=1,
        previous_winners=np.array([r2_lead, r3_lead, r4_lead, r5_lead]),
        sim_func=n_trick_sim,
        verbose=verbose,
    )
    r6_hand, r6_lead = find_best_response(
        lead=r5_lead,
        hands=r5_hand,
        tricks=1,
        previous_winners=np.array([r2_lead, r3_lead, r4_lead, r5_lead], dtype=np.int64),
        best_opener=best_opener_r5,
    )


    result = np.sum(np.array([r2_lead, r3_lead, r4_lead, r5_lead, r6_lead]) % 2)

    if verbose:
        print("Starting hands:\n", dealt_hands)
        print("Trick 1:\n", trick_played(dealt_hands, r2_hand),
        print("Trick 1 winner:", r2_lead))
        print("next round hands:\n", r2_hand)

        print("Trick 2:", trick_played(r2_hand, r3_hand))
        print("Trick 2 winner:", r3_lead)
        print("next round hands:\n", r3_hand) 

        print("Trick 3:\n", trick_played(r3_hand, r4_hand))
        print("Trick 3 winner:", r4_lead)           
        print("next round hands:\n", r4_hand) 

        print("Trick 4:\n", trick_played(r4_hand, r5_hand))
        print("Trick 4 winner:", r5_lead)       
        print("next round hands:\n", r5_hand)

        print("Trick 5:\n", trick_played(r5_hand, r6_hand))
        print("Trick 5 winner:", r6_lead)

        print("Final result:", np.array([r2_lead, r3_lead, r4_lead, r5_lead, r6_lead]))
        

    if result >= 3:
        return 0 
    else:        
        return 1
    


In [49]:
definitive_winner(dealt_hands=hands_test, starting_player=1, verbose=True)

[0.20761246 0.41981132 0.19786432 0.05836576 0.13004484]
[0.08290155 0.04761905 0.         0.        ]
[0. 0. 0.]
[0. 0.]
[0.]
Starting hands:
 [[[  0 120]
  [  0 -14]
  [-12   0]
  [  9   0]
  [  0  -9]]

 [[  0 -10]
  [ 10   0]
  [-11   0]
  [ 14   0]
  [  0 140]]

 [[ 12   0]
  [  0 110]
  [-14   0]
  [ 13   0]
  [  0 130]]

 [[  0 -12]
  [  0 135]
  [ 11   0]
  [  0 100]
  [  0  90]]]
Trick 1 winner: 1
Trick 1:
 [[ 9  0]
 [14  0]
 [12  0]
 [11  0]] None
next round hands:
 [[[  0 120]
  [  0 -14]
  [-12   0]
  [  0  -9]]

 [[  0 -10]
  [ 10   0]
  [-11   0]
  [  0 140]]

 [[  0 110]
  [-14   0]
  [ 13   0]
  [  0 130]]

 [[  0 -12]
  [  0 135]
  [  0 100]
  [  0  90]]]
Trick 2: [[-12   0]
 [-11   0]
 [-14   0]
 [  0 100]]
Trick 2 winner: 3
next round hands:
 [[[  0 120]
  [  0 -14]
  [  0  -9]]

 [[  0 -10]
  [ 10   0]
  [  0 140]]

 [[  0 110]
  [ 13   0]
  [  0 130]]

 [[  0 -12]
  [  0 135]
  [  0  90]]]
Trick 3:
 [[  0  -9]
 [  0 -10]
 [  0 130]
 [  0 -12]]
Trick 3 winner: 2
nex

0